# Optional: rebuild analysis tables from processed source files
The six analysis notebooks use the supplied Parquet files directly. Run this notebook only to rebuild those inputs from the trusted upstream processed pickle files in `../UFallV4/OUTPUTS`.

This is a table builder, not a complete raw-sensor/SAM3/CLIP pipeline. The upstream model inference and synchronisation code is not included in this folder. Rebuilding replaces the corresponding analysis inputs; select the source directory deliberately.

In [ ]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from helpers.bout_features import load_processed_pickle
from helpers.relational import build_relational_tables
REBUILD=False
DATASETS=[
    (Path('../UFallV4/OUTPUTS'),Path('derived'),{'participants','bouts','frames','frame_embeddings'}),
    (Path('../UFallV4/OUTPUTS/single_bout_all_frame'),Path('derived_eye_tracking'),{'bouts','frames','frame_objects'}),
]

In [ ]:
if REBUILD:
    for source_dir,target_dir,keep in DATASETS:
        paths=sorted(source_dir.glob('*_processed.pkl'))
        if not paths: raise FileNotFoundError(source_dir)
        target_dir.mkdir(exist_ok=True)
        writers={}
        try:
            for path in tqdm(paths,desc=str(target_dir)):
                tables=build_relational_tables(load_processed_pickle(path),image_width=1088,image_height=1080)
                for name in sorted(keep):
                    table=tables.get(name)
                    if table is None or table.empty: continue
                    arrow=pa.Table.from_pandas(table,preserve_index=False)
                    if name not in writers:
                        writers[name]=pq.ParquetWriter(target_dir/f'{name}.building.parquet',arrow.schema)
                    if not arrow.schema.equals(writers[name].schema,check_metadata=False):
                        arrow=arrow.cast(writers[name].schema)
                    writers[name].write_table(arrow)
        finally:
            for writer in writers.values(): writer.close()
        for name in writers:
            temporary=target_dir/f'{name}.building.parquet'
            pq.ParquetFile(temporary)  # Check the closed footer before replacing input.
            temporary.replace(target_dir/f'{name}.parquet')
else:
    print('Using supplied data. Set REBUILD=True only to rebuild from upstream sources.')